In [1]:
import pandas as pd
import numpy as np
import pyfixest as pf

panel = pd.read_csv("../data/processed/panel_gdp.csv")
gdp = pd.read_csv("../data/processed/gdp.csv")
g24 = gdp[gdp["year"] == 2024][["geo", "gdp_pc"]].rename(columns={"gdp_pc": "gdp_2024"})
panel = panel.merge(g24, on="geo", how="left")

In [2]:
def beta(data, y="ln_s_may", x="ln_e_2025", gdp="gdp_pc", fe="isco2", slopes=False):
    d = data.dropna(subset=[y, x]).copy()
    d["country_id"] = d["geo"].astype("category").cat.codes
    rhs = x
    if slopes:
        d["lg"] = np.log(d[gdp])
        cols = []
        for g in sorted(d["isco1"].unique())[1:]:
            d["gdp_x_" + g] = d["lg"] * (d["isco1"] == g)
            cols.append("gdp_x_" + g)
        rhs = x + " + " + " + ".join(cols)
    m = pf.feols(f"{y} ~ {rhs} | geo + {fe}", data=d, vcov={"CRV1": "country_id"})
    b, se = m.coef()[x], m.se()[x]
    return {"beta": round(b, 3), "se": round(se, 3), "t_vs_1": round((b - 1) / se, 1),
            "cells": m._N, "countries": d["geo"].nunique()}

In [3]:
main = panel[panel["main_sample"]]

# ISCO 1-digit: add up the 2-digit shares inside each broad group
one = main.groupby(["geo", "isco1"], as_index=False).agg(
    s_may=("s_may", "sum"), e_2025=("e_2025", "sum"), gdp_pc=("gdp_pc", "first"))
one["ln_s_may"] = np.log(one["s_may"])
one["ln_e_2025"] = np.log(one["e_2025"])

# small countries: fewer than 4 million employed in 2025
emp = pd.read_csv("../data/processed/employment_shares.csv")
total = emp[emp["year"] == 2025].groupby("geo")["value"].sum() / 1000
small = [g for g in main["geo"].unique() if total[g] < 4]
print(one.shape, small)

(128, 7) ['DK', 'FI', 'IE']


In [4]:
codes = {"AUT": "AT", "BEL": "BE", "CZE": "CZ", "DNK": "DK", "FIN": "FI", "FRA": "FR",
         "DEU": "DE", "GRC": "EL", "IRL": "IE", "ITA": "IT", "NLD": "NL", "POL": "PL",
         "PRT": "PT", "ROU": "RO", "ESP": "ES", "SWE": "SE",
         "BGR": "BG", "HUN": "HU", "LTU": "LT", "SVK": "SK"}

uw = pd.read_csv("../data/processed/usage_shares_usw.csv")
uw = uw[(uw["date_start"] == "2026-05-01") & (uw["geo_id"].isin(codes))].copy()
uw["geo"] = uw["geo_id"].map(codes)
uw["ln_s_may_us"] = np.log(uw["pct_isco"] / 100)
main_us = main.merge(uw[["geo", "isco2", "ln_s_may_us"]], on=["geo", "isco2"], how="left")

In [7]:
epi = pd.read_csv("../data/raw/english/ef_epi_2025.csv")
main_en = main.merge(epi, on="geo", how="inner")
print(len(main_en), main_en["geo"].nunique())

458 15


In [8]:
main = panel[panel["main_sample"]]
checks = {
    "Main": dict(data=main),
    "EU >= 70% (20 countries)": dict(data=panel),
    "April usage": dict(data=main, y="ln_s_apr"),
    "2024 employment": dict(data=main, x="ln_e_2024"),
    "Drop ISCO 25 and 23": dict(data=main[~main["isco2"].isin(["OC25", "OC23"])]),
    "Drop Ireland": dict(data=main[main["geo"] != "IE"]),
    "2024 GDP": dict(data=main, gdp="gdp_2024"),
    "ISCO 1-digit": dict(data=one, fe="isco1"),
    "Drop small countries": dict(data=main[~main["geo"].isin(small)]),
    "US-weight crosswalk": dict(data=main_us, y="ln_s_may_us"),
    "English slopes (no Ireland)": dict(data=main_en, gdp="epi"),
}

rows = []
for name, args in checks.items():
    for s in [False, True]:
        r = beta(**args, slopes=s)
        r["check"] = name
        r["slopes"] = s
        rows.append(r)

rob = pd.DataFrame(rows)[["check", "slopes", "beta", "se", "t_vs_1", "cells", "countries"]]
rob.to_csv("../output/tables/robustness.csv", index=False)
rob

C:\Research\AI Exposure in the European labour market\.venv\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3 singleton fixed effect(s) dropped from the model.
  warnings.warn(
C:\Research\AI Exposure in the European labour market\.venv\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3 singleton fixed effect(s) dropped from the model.
  warnings.warn(


,check,slopes,beta,se,t_vs_1,cells,countries
0,Main,False,0.036,0.035,-27.4,482,16
1,Main,True,0.030,0.038,-25.8,482,16
2,EU >= 70% (20 countries),False,0.034,0.030,-32.0,575,20
3,EU >= 70% (20 countries),True,0.015,0.034,-28.8,575,20
4,April usage,False,0.048,0.024,-40.1,452,16
5,April usage,True,0.044,0.023,-41.9,452,16
6,2024 employment,False,0.039,0.038,-25.2,482,16
7,2024 employment,True,0.034,0.041,-23.7,482,16
8,Drop ISCO 25 and 23,False,0.036,0.036,-27.1,450,16
9,Drop ISCO 25 and 23,True,0.030,0.037,-25.9,450,16
